In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from scipy.stats import entropy
from scipy.stats import multivariate_normal
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import confusion_matrix,average_precision_score,roc_curve,auc,precision_score,recall_score
from sklearn.covariance import LedoitWolf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df=pd.read_csv('/content/ecoli.csv')
df.shape

(336, 8)

In [ ]:
y=df.iloc[:,-1]
unique, counts = np.unique(y, return_counts=True)
dict_X = dict(zip(unique, counts))
dict_X

{'inlier': np.int64(327), 'outlier': np.int64(9)}

In [ ]:
from sklearn import preprocessing
for x in df.columns:
    while df[x].dtype == 'object':
        lbl = preprocessing.LabelEncoder()
        lbl.fit(list(df[x].values))
        df[x] = lbl.transform(list(df[x].values))

In [ ]:
data=df.iloc[:,:-1]
dataset=data.copy()
y_true=df.iloc[:,-1]

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
sc=scaler.fit(data)
sc_x=scaler.transform(data)

In [ ]:
#distance-based nearest neighbor: EDNN
distances = pdist(data, metric='euclidean')

# convert distances to a square matrix
d_matrix = squareform(distances)

In [ ]:
##distance-based nearest neighbor: EDNN_define outlierness

k=41
d_sort = np.argsort(d_matrix, axis=1)[:,:k]
#idx

d_res = np.take_along_axis(d_matrix, d_sort, axis=1)
d_val=pd.DataFrame(d_res)
dataset['out_d']=np.sum(d_val,axis=1)

In [ ]:
#entropy-based nearest neighbor
m = np.mean(data, axis=0)
c = np.cov(data.T)
mvn = multivariate_normal(mean=m, cov=c)

# calculate the probability of each row in the DataFrame
prob = mvn.pdf(data)
dt=data.copy()
dt['prob']=prob+np.max(prob)

In [ ]:
# mean vector
m = np.mean(data, axis=0)

# shrinkage covariance estimation
lw = LedoitWolf()
lw.fit(data)

# robust covariance matrix
c_reg = lw.covariance_

# multivariate normal model
mvn = multivariate_normal(mean=m, cov=c_reg)

# probability density
prob = mvn.pdf(data)

# append probability
dt = data.copy()
dt['prob'] = prob + np.max(prob)

In [ ]:
def row_plus_df(row, df):
    return -np.log(row.values/df.values)

# Apply the row_plus_df function to each row of the DataFrame
t_matrix = dt[['prob']].apply(row_plus_df, df=dt[['prob']], axis=1)

In [ ]:
listb = []
for i in t_matrix.index:
    lista = []
    for j in t_matrix.iloc[i]:
        lista.extend(j)
    listb.append(lista)
e_matrix = np.array(listb)

In [ ]:
##entropy-based nearest neighbor: define outlierness

k=41
e_sort = np.argsort(e_matrix, axis=1)[:, ::-1][:,:k]
#e_sort = np.argsort(e_matrix, axis=1)[:,:k]

e_res = np.take_along_axis(e_matrix, e_sort, axis=1)
e_val=pd.DataFrame(e_res)
dataset['out_e']=np.sum(e_val,axis=1)

In [ ]:
X = np.array(data)

# Ledoit-Wolf shrinkage covariance
lwm = LedoitWolf()
lwm.fit(data)

# inverse robust covariance
inv_cov_reg = np.linalg.inv(lwm.covariance_)

# Mahalanobis distances
dist_m = pdist(X, metric='mahalanobis', VI=inv_cov_reg)

# square distance matrix
m_matrix = squareform(dist_m)

In [ ]:
#distance-based nearest neighbor
dist_m = pdist(data, metric='Mahalanobis')

# convert distances to a square matrix
m_matrix = squareform(dist_m)

In [ ]:
##distance-based nearest neighbor: define outlierness

k=41
m_sort = np.argsort(m_matrix, axis=1)[:,:k]
#idx

m_res = np.take_along_axis(m_matrix, m_sort, axis=1)
m_val=pd.DataFrame(m_res)
dataset['out_m']=np.sum(m_val,axis=1)

In [ ]:
dft=dataset[['out_d']]

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
dft_x=scaler.fit(dft)
dft_x=scaler.transform(dft)

In [ ]:
dt_p=pd.DataFrame(dft_x)
dt_r=pd.DataFrame(np.max(dt_p,axis=1))

In [ ]:
def sigmoid(x):
    return (1 / (1 + np.exp(-x)))

# Apply the sigmoid function to the features
sig = sigmoid(dt_r)
y_score=sig

In [ ]:
#OD results
fpr_od_1, tpr_od_1, thresholds_od = roc_curve(y_true, y_score,pos_label=1)
#print('Precision: %.4f' % precision_score(y_true, y_pred))
#print('Recall: %.4f' % recall_score(y_true, y_pred))
print('Average prcision: %.4f' % average_precision_score(y_true, y_score))
print('AUC(PRC): %.4f'% auc(fpr_od_1,tpr_od_1))

Average prcision: 0.6169
AUC(PRC): 0.8855


In [ ]:
result=data.copy()

In [ ]:
#KNN model
from pyod.models.knn import KNN
kn=KNN(metric='euclidean',n_neighbors=40)
kn.fit(data)

KNN(algorithm='auto', contamination=0.1, leaf_size=30, method='largest',
  metric='euclidean', metric_params=None, n_jobs=1, n_neighbors=40, p=2,
  radius=1.0)

In [ ]:
#KNN confusion matrix
knn=kn.labels_
result['Outlier_kn']=knn
y_pred_kn=result[['Outlier_kn']]
y_score_kn=kn.predict_proba(data)[:,1]
confusion_matrix(y_true,y_pred_kn)

array([[300,  27],
       [  2,   7]])

In [ ]:
#COF results
fpr_kn, tpr_kn, thresholds_kn = roc_curve(y_true, y_score_kn,pos_label=1)
#print('Precision: %.3f' % precision_score(y_true, y_pred_kn))
#print('Recall: %.3f' % recall_score(y_true, y_pred_kn))
print('Average Precision: %.4f' % average_precision_score(y_true, y_score_kn))
print('AUC(PRC): %.4f'% auc(fpr_kn,tpr_kn))

Average Precision: 0.6137
AUC(PRC): 0.8960


In [ ]:
#COF model
from pyod.models.lof import LOF
lof=LOF()
lof.fit(data)

LOF(algorithm='auto', contamination=0.1, leaf_size=30, metric='minkowski',
  metric_params=None, n_jobs=1, n_neighbors=20, novelty=True, p=2)

In [ ]:
lf=lof.predict(data)
result['Outlier_lof']=lf
y_pred_lof=result[['Outlier_lof']]
y_score_lof=lof.predict_proba(data)[:,1]
confusion_matrix(y_true,y_pred_lof)

array([[304,  23],
       [  2,   7]])

In [ ]:
#LOF results
fpr_lof, tpr_lof, thresholds_lof = roc_curve(y_true, y_score_lof,pos_label=1)
#print('Precision: %.3f' % precision_score(y_true, y_pred_lof))
#print('Recall: %.3f' % recall_score(y_true, y_pred_lof))
print('Average Precision: %.4f' % average_precision_score(y_true, y_score_lof))
print('AUC(PRC): %.4f'% auc(fpr_lof,tpr_lof))

Average Precision: 0.4650
AUC(PRC): 0.8566


In [ ]:
#COF model
from pyod.models.cof import COF
oc=COF()
oc.fit(data)

COF(contamination=0.1, method='fast', n_neighbors=20)

In [ ]:
#COF confusion matrix
ocs=oc.labels_
result['Outlier_oc']=ocs
y_pred_oc=result[['Outlier_oc']]
y_score_oc=oc.predict_proba(data)[:,1]
confusion_matrix(y_true,y_pred_oc)

array([[300,  27],
       [  2,   7]])

In [ ]:
#COF results
fpr_oc, tpr_oc, thresholds_oc = roc_curve(y_true, y_score_oc,pos_label=1)
#print('Precision: %.3f' % precision_score(y_true, y_pred_oc))
#print('Recall: %.3f' % recall_score(y_true, y_pred_oc))
print('Average Precision: %.4f' % average_precision_score(y_true, y_score_oc))
print('AUC(PRC): %.4f'% auc(fpr_oc,tpr_oc))

Average Precision: 0.1837
AUC(PRC): 0.8481


In [ ]:
#ABOD
from pyod.models.abod import ABOD
ab=ABOD(n_neighbors=20)
ab.fit(data)

ABOD(algorithm='auto', contamination=0.1, leaf_size=30, method='fast',
   metric='minkowski', metric_params=None, n_jobs=1, n_neighbors=20, p=2)

In [ ]:
abod=ab.predict(data)
result['Outlier_ab']=abod
y_pred_ab=result[['Outlier_ab']]
y_score_ab=ab.predict_proba(data)[:,1]
confusion_matrix(y_true,y_pred_ab)

array([[300,  27],
       [  2,   7]])

In [ ]:
#LOF results
fpr_ab, tpr_ab, thresholds_ab = roc_curve(y_true, y_score_ab,pos_label=1)
#print('Precision: %.3f' % precision_score(y_true, y_pred_lof))
#print('Recall: %.3f' % recall_score(y_true, y_pred_lof))
print('Average Precision: %.4f' % average_precision_score(y_true, y_score_ab))
print('AUC(PRC): %.4f'% auc(fpr_ab,tpr_ab))

Average Precision: 0.3638
AUC(PRC): 0.8617


In [ ]:
#DBSCAN
from sklearn.cluster import DBSCAN
db=DBSCAN(eps=0.5, min_samples=5).fit(sc_x)
db.fit(data)

DBSCAN()

In [ ]:
from sklearn.neighbors import NearestNeighbors

eps = 0.5
min_samples = 5
# Compute decision values: number of neighbors within eps
def dbscan_decision_values(data, eps):
    nbrs = NearestNeighbors(radius=eps).fit(data)
    # radius_neighbors returns all neighbors within given radius
    radius_neighbors = nbrs.radius_neighbors(data, return_distance=False)
    decision_values = np.array([len(neigh) for neigh in radius_neighbors])
    return decision_values

decision_values = dbscan_decision_values(data, eps)

In [ ]:
def sigmoid(x):
    return (1 / (1 + np.exp(-x)))

# Apply the sigmoid function to the features
y_sig = (1-sigmoid(decision_values))

In [ ]:
#OD results
fpr_db, tpr_db, thresholds_db = roc_curve(y_true, y_sig,pos_label=1)
#print('Precision: %.4f' % precision_score(y_true, y_pred))
#print('Recall: %.4f' % recall_score(y_true, y_pred))
print('Average prcision: %.4f' % average_precision_score(y_true, y_sig))
print('AUC(PRC): %.4f'% auc(fpr_db,tpr_db))

Average prcision: 0.5767
AUC(PRC): 0.8818


In [ ]:
from sklearn.cluster import MeanShift
from sklearn.metrics import pairwise_distances_argmin_min
MS = MeanShift(bandwidth=2).fit(data)

In [ ]:
labels = MS.labels_
cluster_centers = MS.cluster_centers_

In [ ]:
# Compute decision values: inverse of distance to nearest cluster center
closest_center_indices, distances = pairwise_distances_argmin_min(data, cluster_centers)

# Normalize and invert distances: higher = closer to cluster center (more confident)
decision_values_ms = 1 - (distances / distances.max())


In [ ]:
# Apply the sigmoid function to the features
y_sig_ms = (1-sigmoid(decision_values_ms))

#OD results
fpr_ms, tpr_ms, thresholds_ms = roc_curve(y_true, y_sig_ms,pos_label=1)
#print('Precision: %.4f' % precision_score(y_true, y_pred))
#print('Recall: %.4f' % recall_score(y_true, y_pred))
print('Average prcision: %.4f' % average_precision_score(y_true, y_sig_ms))
print('AUC(PRC): %.4f'% auc(fpr_ms,tpr_ms))

Average prcision: 0.4359
AUC(PRC): 0.8675


In [ ]:
#ROC analysis
import plotly.graph_objects as go

fig = go.Figure()
fig.add_shape(
    type='line', line=dict(dash='dash'),
    x0=0, x1=1, y0=0, y1=1)

fig.add_trace(go.Scatter(x=fpr_od_1, y=tpr_od_1, name=f'k=5(AUC={auc(fpr_od_1, tpr_od_1):.3f})', mode='lines'))
fig.add_trace(go.Scatter(x=fpr_od_2, y=tpr_od_2, name=f'k=10(AUC={auc(fpr_od_2, tpr_od_2):.3f})', mode='lines'))
fig.add_trace(go.Scatter(x=fpr_od_3, y=tpr_od_3, name=f'k=15(AUC={auc(fpr_od_3, tpr_od_3):.3f})', mode='lines'))
fig.add_trace(go.Scatter(x=fpr_od_4, y=tpr_od_4, name=f'k=20(AUC={auc(fpr_od_4, tpr_od_4):.3f})', mode='lines'))
fig.add_trace(go.Scatter(x=fpr_od_5, y=tpr_od_5, name=f'k=25(AUC={auc(fpr_od_5, tpr_od_5):.3f})', mode='lines'))
fig.add_trace(go.Scatter(x=fpr_od_6, y=tpr_od_6, name=f'k=30(AUC={auc(fpr_od_6, tpr_od_6):.3f})', mode='lines'))
fig.add_trace(go.Scatter(x=fpr_od_7, y=tpr_od_7, name=f'k=35(AUC={auc(fpr_od_7, tpr_od_7):.3f})', mode='lines'))
fig.add_trace(go.Scatter(x=fpr_od_8, y=tpr_od_8, name=f'k=40(AUC={auc(fpr_od_8, tpr_od_8):.3f})', mode='lines'))
fig.add_trace(go.Scatter(x=fpr_od_9, y=tpr_od_9, name=f'k=45(AUC={auc(fpr_od_9, tpr_od_9):.3f})', mode='lines',marker_color='hotpink'))
fig.add_trace(go.Scatter(x=fpr_od_10, y=tpr_od_10, name=f'k=50(AUC={auc(fpr_od_10, tpr_od_10):.3f})', mode='lines',marker_color='crimson'))


fig.update_layout(
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    yaxis=dict(scaleanchor="x", scaleratio=1),
    xaxis=dict(constrain='domain'),
    width=700, height=600)
fig.show()

In [ ]:
#df['Out_en']=y_pred
df['Out_lf']=result[['Outlier_lof']]

In [ ]:
df['Out']=np.sum(df[['Outlier','Out_lf']],axis=1)
df['Out'] = df[['Out']].apply(lambda row: row['Out']==2,axis=1).astype(int)


In [ ]:
import plotly.express as px
fig=px.scatter_3d(df, x='V1',y='V6',z='V36',color='outlier')
fig.update_traces(marker=dict(size=7,
                              line=dict(width=3,
                                        color='MediumPurple')),
                  selector=dict(mode='markers'))
fig.show()

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=0)
projections = tsne.fit_transform(data)

In [ ]:
ts=pd.DataFrame(projections)
ts.columns=['tsne_feature1','tsne_feature2']

In [ ]:
y_out=df['Out'].astype('category')
ts['outlier']=y_out

In [ ]:
import plotly.express as px
fig = px.scatter(ts, x='tsne_feature1', y='tsne_feature2',color='outlier')
fig.show()

In [ ]:
import plotly.figure_factory as ff
colors = ['magenta', 'yellow','green', 'red']
fig = ff.create_distplot([dt_p[c] for c in dt_p.columns], dt_p.columns, colors=colors,show_hist=False,bin_size=0.25)
fig.show()